# LSTM-Baseline (Count) – Colab-Runner

Repo `bike-link-prediction` nach Google Drive hochladen. Struktur:
```
bike-link-prediction/
├── evaluation/shared_eval.py
├── prepared Data/   (graphmixer_edges.csv …)
└── lstm/            (lstm_count.py, dieses Notebook)
```

## 1. GPU prüfen

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Google Drive einbinden

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Pfade setzen (ROOT anpassen)

In [ ]:
import os
ROOT = "/content/drive/MyDrive/bike-link-prediction"
LSTM_DIR = os.path.join(ROOT, "lstm")
EVAL_DIR = os.path.join(ROOT, "evaluation")
PREP_DIR = os.path.join(ROOT, "prepared Data")
for p in [LSTM_DIR, EVAL_DIR, PREP_DIR]:
    print(("OK  " if os.path.isdir(p) else "FEHLT ") + p)
fp = os.path.join(PREP_DIR, "graphmixer_edges.csv"); print(("OK  " if os.path.isfile(fp) else "FEHLT ") + fp)

## 4. In den lstm-Ordner wechseln

In [ ]:
%cd "$LSTM_DIR" 

## 5. Konfiguration

In [ ]:
from lstm_count import LSTMConfig
cfg = LSTMConfig(epochs=10)   # LSTMConfig(epochs=2) für Smoke-Test
print("Lookback:", cfg.lookback, "| hidden:", cfg.hidden_dim, "| epochs:", cfg.epochs)

## 6. Training + Bewertung

In [ ]:
from lstm_count import main
main(cfg)

## 7. Ergebnisse erneut bewerten

In [ ]:
import sys, pandas as pd
sys.path.insert(0, EVAL_DIR)
from shared_eval import SharedLinkEval
ev = SharedLinkEval()
for split in ["val", "test"]:
    pred = pd.read_csv(f"predictions/lstm_pred_{split}.csv")
    r = ev.score_count(pred, split=split)
    print(f"[{split}] MSE={r['mse']:.4f} MAE={r['mae']:.4f} RMSE={r['rmse']:.4f}")